# 07 — Final Evaluation

**CineMind Phase 1 · Step 7** — Compare every ranker on the same held-out test
set, produce a results table, and show example recommendations.

Loads `artifacts/two_tower.pt` and re-evaluates three rankers:
popularity baseline, two-tower, and the hybrid (two-tower + 0.30 × log-popularity).

**Requires:** `artifacts/two_tower.pt`, `artifacts/item_vecs.npy`,
`data/train_positives.csv`, `data/test_positives.csv`, `data/items.csv`

**Output:** `artifacts/results.csv`

In [1]:
import os, sys

# Run from project root regardless of where Jupyter was launched
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Make src/ importable
if not any('src' in p for p in sys.path):
    sys.path.insert(0, 'src')

import cinemind_utils as cu
print("cwd:", os.getcwd())

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
cu.ensure_dirs()

cwd: c:\Users\shaur\Desktop\My codes shaurya\Content Recommendation system\cinemind _phase1\cinemind_phase1


## 1 · Re-define model classes

The `TwoTower` class is defined here so this notebook is fully self-contained
(Python cannot directly `import` a module whose filename starts with a digit).

In [2]:
class Tower(nn.Module):
    def __init__(self, in_dim, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Linear(128, out_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, genre_t, item_counts, id_dim=64, out_dim=64):
        super().__init__()
        self.user_emb   = nn.Embedding(n_users, id_dim)
        self.item_emb   = nn.Embedding(n_items, id_dim)
        self.user_tower = Tower(id_dim, out_dim)
        self.item_tower = Tower(id_dim + 19, out_dim)
        self.temp       = 0.07
        self.register_buffer("genre_t", genre_t)
        logq = torch.log(torch.tensor(item_counts / item_counts.sum() + 1e-9))
        self.register_buffer("logq", logq.float())

    def user_vec(self, u):
        return self.user_tower(self.user_emb(u))

    def item_vec(self, i):
        return self.item_tower(torch.cat([self.item_emb(i), self.genre_t[i]], dim=-1))

## 2 · Load data and model

In [3]:
train     = pd.read_csv("data/train_positives.csv")
test      = pd.read_csv("data/test_positives.csv")
items     = pd.read_csv("data/items.csv")
item_vecs = np.load("artifacts/item_vecs.npy").astype(np.float32)

n_items = item_vecs.shape[0]
n_users = int(max(train.user_id.max(), test.user_id.max())) + 1

genre_mat = np.zeros((n_items, 19), dtype=np.float32)
genre_mat[items.movie_id.values] = items[[f"g{i}" for i in range(19)]].to_numpy(dtype=np.float32)

pop_cnt     = train.movie_id.value_counts().reindex(range(n_items), fill_value=0).to_numpy()
item_counts = pop_cnt.astype(np.float64) + 1.0

model = TwoTower(n_users, n_items, torch.tensor(genre_mat), item_counts)
model.load_state_dict(torch.load("artifacts/two_tower.pt", map_location="cpu"))
model.eval()

ivecs = torch.tensor(item_vecs)
print(f"Model loaded.  n_users={n_users}  n_items={n_items}")

Model loaded.  n_users=944  n_items=1683


## 3 · Define rankers

In [4]:
title_of = dict(zip(items.movie_id, items.title))
seen     = train.groupby("user_id").movie_id.apply(set).to_dict()
truth    = test.groupby("user_id").movie_id.apply(set).to_dict()
top_pop  = train.movie_id.value_counts().index.to_numpy()
log_pop  = np.log1p(pop_cnt); log_pop /= log_pop.max()

def pop_rank(u, k=10):
    s = seen.get(u, set())
    return [m for m in top_pop if m not in s][:k]

def tt_rank(u, k=10):
    with torch.no_grad():
        scores = (model.user_vec(torch.tensor([u])) @ ivecs.t()).squeeze(0).numpy()
    for m in seen.get(u, set()):
        scores[m] = -1e9
    return np.argpartition(-scores, k)[:k]

def hybrid_rank(u, k=10, alpha=0.30):
    with torch.no_grad():
        scores = (model.user_vec(torch.tensor([u])) @ ivecs.t()).squeeze(0).numpy()
    scores += alpha * log_pop
    for m in seen.get(u, set()):
        scores[m] = -1e9
    return np.argpartition(-scores, k)[:k]

## 4 · Evaluate all rankers

In [5]:
rankers = {
    "Popularity baseline":   pop_rank,
    "Two-tower":             tt_rank,
    "Two-tower + pop prior": hybrid_rank,
}

rows = []
print("Evaluating ...")
for name, fn in rankers.items():
    p, r = cu.precision_recall_at_k(fn, truth, k=10)
    rows.append({"model": name, "precision@10": round(p, 4), "recall@10": round(r, 4)})

results = pd.DataFrame(rows)

print(f"\n{'Model':<26}{'Precision@10':>14}{'Recall@10':>14}")
print("-" * 56)
for _, row in results.iterrows():
    print(f"{row['model']:<26}{row['precision@10']:>14.4f}{row['recall@10']:>14.4f}")
print("-" * 56)

Evaluating ...

Model                       Precision@10     Recall@10
--------------------------------------------------------
Popularity baseline               0.0603        0.0665
Two-tower                         0.0581        0.0917
Two-tower + pop prior             0.0945        0.1254
--------------------------------------------------------


## 5 · Save results

In [6]:
results.to_csv("artifacts/results.csv", index=False)
print("Saved artifacts/results.csv")

Saved artifacts/results.csv


## 6 · Example recommendations for one user

In [7]:
u = 196 if 196 in seen else int(train.user_id.iloc[0])

liked = (train[train.user_id == u]
         .sort_values("rating", ascending=False)
         .movie_id.head(5))

print(f"User {u} liked:")
for m in liked:
    print(f"  {title_of.get(m)}")

print(f"\nHybrid recommends:")
for m in hybrid_rank(u):
    if m in title_of:
        print(f"  {title_of.get(m)}")

print("\nPhase 1 complete.")
print("You have: reproduced the baseline, trained a modern model that beats it,")
print("embedded content for cold-start, set up vector search, and compared all")
print("approaches side by side. This is your portfolio evidence.")

User 196 liked:
  English Patient, The (1996)
  Babe (1995)
  Being There (1979)
  Fish Called Wanda, A (1988)
  Stand by Me (1986)

Hybrid recommends:
  Sense and Sensibility (1995)
  Postino, Il (1994)
  Godfather, The (1972)
  Toy Story (1995)
  Fargo (1996)
  Liar Liar (1997)
  When Harry Met Sally... (1989)
  Star Wars (1977)
  Annie Hall (1977)
  Four Weddings and a Funeral (1994)

Phase 1 complete.
You have: reproduced the baseline, trained a modern model that beats it,
embedded content for cold-start, set up vector search, and compared all
approaches side by side. This is your portfolio evidence.
